# GPEC 447 — Project Notebook

## 1. Project Title

**Industrial–Logistics Coupling in the Otay Mesa Border Zone:  
A Spatial Analysis of Freight Infrastructure and Industrial Activity in San Diego**

---

## 2. Team Members

| Name | Student ID |
|------|------------|
| Christian Phouasalith | gpec447sp26_23 |
| Joseph Hurr | gpec447sp26_11 |
| Canyu Li | *add ID* |

> **Note on notebook structure.** This report notebook summarizes the project across all 12 required sections. The detailed analysis lives in three component notebooks that each member can run independently:
> - **`Code_canyu_June_7th.ipynb`** — tract-level analysis (Census ACS + LODES workplace jobs, zoning/land-use overlays, proximity indicators)
> - **`Code_Joseph.ipynb`** — business-point indicators (POE distance, freight-corridor buffers, statistics, maps)
> - **`Code_Christian.ipynb`** — *(in progress — see Section 7 and Section 9 placeholders)*


## 3. Research Question & Importance

### Question
Is industrial activity in the Otay Mesa border area more strongly associated with ports of entry (POEs) and major freight corridors than industrial activity in comparable non-border industrial areas of San Diego?

### Intended Audience
Regional planners, economic development agencies (SANDAG, City of San Diego), cross-border logistics operators, and policymakers considering infrastructure investments around the proposed Otay Mesa East port of entry.

### Business Case
San Diego's Otay Mesa district sits at one of the busiest commercial border crossings in the world. Understanding whether industrial clustering there is genuinely organized around port access — rather than simply reflecting leftover land use patterns — has direct implications for:
- Land use planning decisions near the proposed Otay Mesa East POE
- Infrastructure investment prioritization for freight corridors
- Economic development strategies targeting cross-border logistics

### Evolution from Proposal
The core question remained consistent with the project proposal. The primary methodological evolution was a shift from industrial zone polygons as the unit of analysis to two complementary units — census tracts (Canyu) and geocoded business license records (Joseph and Christian) — following feedback that San Diego's industrial zoning reflects historical land allocation rather than deliberate freight-access decisions. A second evolution, in response to instructor feedback, was replacing ESRI GeoEnrichment with Census LODES workplace-jobs data for the tract-level activity measure.


## 4. Background & Literature

### References

1. **SANDAG FreightViewer** — Interactive visualization of San Diego's freight network, ports of entry, and major roads.  
   [https://gis.sandag.org/FreightViewer/](https://gis.sandag.org/FreightViewer/)  
   *Provided the freight infrastructure reference layers (POE points, 45 major road segments) used in both the tract-level and business-point analyses.*

2. **US Census Bureau — American Community Survey (ACS) 5-Year Estimates, 2019–2023 (DP03)**  
   [https://www.census.gov/programs-surveys/acs](https://www.census.gov/programs-surveys/acs)  
   *Industry-of-employment counts by tract. Clarified that ACS measures where industrial workers* live*, not where they* work* — motivating the addition of LODES as the primary activity measure.*

3. **US Census LEHD LODES8 — Workplace Area Characteristics (WAC)**  
   [https://lehd.ces.census.gov/data/](https://lehd.ces.census.gov/data/)  
   *Jobs by 2-digit NAICS sector at the workplace location. Adopted as the primary tract-level activity measure because it captures where industrial activity is physically located.*

4. **City of San Diego — Zoning and Parcel Information Portal (ZAPP)**  
   [https://www.arcgis.com/apps/instant/sidebar/index.html?appid=75f6a5d68aee481f8ff48240bcaa1239](https://www.arcgis.com/apps/instant/sidebar/index.html?appid=75f6a5d68aee481f8ff48240bcaa1239)  
   *Revealed that San Diego's industrial zoning reflects "leftover land" allocation rather than freight-access optimization — the key limitation that motivated the shift from zone polygons to business and employment data.*

5. **US Census TIGER/Line Cartographic Boundary Files, 2023**  
   [https://www2.census.gov/geo/tiger/GENZ2023/](https://www2.census.gov/geo/tiger/GENZ2023/)  
   *Census tract geometries (736 San Diego County tracts), the spatial unit for the tract-level analysis and the SD County clip boundary for the business-point analyses.*

6. **GADM — Global Administrative Areas, USA Level 2**  
   [https://gadm.org/](https://gadm.org/)  
   *San Diego County administrative boundary used for spatial filtering.*

### How References Shaped the Analysis
The ZAPP zoning layer directly prompted the shift away from industrial zone polygons. The contrast between ACS (residence-based) and LODES (workplace-based) employment measures led to using LODES as the primary activity indicator. SANDAG's freight network data provided the common infrastructure reference layer across all three component analyses.


## 5. Python Packages

The packages below are imported and described in the code cell that follows.  
**Evolution from proposal:** `arcgis.geoenrichment` is no longer required for the tract-level analysis — Census LODES provides equivalent workplace-jobs data without ESRI dependency (it remains available as an optional cross-check only). `scipy.stats` was added to support statistical significance testing not anticipated in the proposal.


In [ ]:
# ── Spatial analysis ──────────────────────────────────────────────────────────
import geopandas as gpd        # Spatial dataframes, CRS transformation, spatial joins
import pandas as pd            # Tabular data manipulation and aggregation
import numpy as np             # Numerical operations and array math

# ── Geometry ──────────────────────────────────────────────────────────────────
from shapely.geometry import Point, box   # Point and polygon construction

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt           # Static maps and charts
import matplotlib.patches as mpatches     # Filled legend patches
from matplotlib.lines import Line2D       # Custom legend line elements

# ── Statistical testing ───────────────────────────────────────────────────────
from scipy.stats import mannwhitneyu      # Non-parametric group comparison
from scipy.stats import chi2_contingency  # Proportion test for binary outcomes
from scipy.stats import gaussian_kde      # Kernel density estimation for heatmaps

# ── Network requests ──────────────────────────────────────────────────────────
import requests                # Fetching GeoJSON / LODES layers from public APIs
from io import BytesIO         # In-memory file handling for remote files

import warnings
warnings.filterwarnings("ignore")

print("All packages imported successfully.")


## 6. Data Sources

| Source | URL | Description |
|--------|-----|-------------|
| SD Business License Records | Internal (City of San Diego) | Active business license records filtered to industrial NAICS |
| Census TIGER Tracts 2023 | [Link](https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_06_tract_500k.zip) | San Diego County census tract geometries (736 tracts) |
| ACS 5-Year 2019–2023 (DP03) | [Link](https://api.census.gov/data/2022/acs/acs5/profile) | Tract-level industry-of-employment (resident-based, context) |
| LEHD LODES8 WAC | [Link](https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/) | Workplace jobs by 2-digit NAICS sector (primary activity measure) |
| SANDAG POE Points | [Link](https://gis.sandag.org/FreightViewer/data/poe_points_update.json) | Port of entry point locations |
| SANDAG Major Roads | [Link](https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json) | Major freight road network (45 features) |
| SD Community Planning Districts | Internal / SanGIS | Official city planning area boundaries (`Community_Plan_SD.geojson`) |
| SD Zoning Base | [Link](https://geo.sandag.org/server/rest/directories/downloads/Zoning_Base_SD.geojson) | All city base zone polygons |
| SD General Plan Land Use | SanGIS `General_Plan_Land_Use_SD` | General plan land-use designations |
| GADM USA Level 2 | [Link](https://gadm.org/) | County administrative boundary |

### Evolution from Proposal
The proposal relied on industrial zone polygons. These were replaced by (a) geocoded business license records and (b) tract-level LODES workplace jobs. The ACS API is retained as context only. ESRI GeoEnrichment was dropped as a dependency.

### Data Quality Concerns
- **Geocoding validity:** Geocode scores (80–100) indicate address-matching confidence, not whether the registered address is an operational freight facility. A sample address-validity worksheet is included in the tract notebook (Section 9c).
- **ACS vs. LODES:** ACS counts industrial workers by residence; LODES counts jobs by workplace. LODES is used as the primary activity measure for that reason.
- **SANDAG roads:** The major roads layer dates to 2017.
- **Desired but unavailable:** Tijuana-side industrial data; parcel-level operational land use; drive-time network distances to POEs.


In [ ]:
# Data sources load live from URLs throughout the component notebooks.
# Documented here for reproducibility and zip-archive inclusion.

DATA_SOURCES = {
    "census_tracts":  "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_06_tract_500k.zip",
    "acs_api":        "https://api.census.gov/data/2022/acs/acs5/profile",
    "lodes_wac":      "https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/",
    "sandag_poe":     "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
    "sandag_roads":   "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
    "sd_zoning_base": "https://geo.sandag.org/server/rest/directories/downloads/Zoning_Base_SD.geojson",
}

for name, url in DATA_SOURCES.items():
    print(f"{name:16s}: {url}")


## 7. Data Cleaning

Cleaning was performed across the three component analyses. More wrangling was required than the proposal anticipated, primarily due to NAICS code formatting in the raw business data and the discovery that some geocoded businesses fell outside San Diego County.

### Tract-level (Canyu)
- Corrected ACS denominator to DP03_0032E (civilian employed 16+); cast all employment columns to numeric.
- Aggregated LODES block-level jobs to tracts via the 15-digit `w_geocode` → 11-digit GEOID crosswalk.
- Built study-area labels from a fixed `CPNAME → label` map on Community Plan districts.

### Business-point (Joseph)
- Standardized column names; cleaned NAICS codes (removed `.0` float artifacts).
- Filtered to Core industrial NAICS; removed records outside SD County and non-matched geocodes.
- Assigned subareas via spatial join on Community Plan districts.

### ⚠️ Christian — pending fixes flagged in team review
> *Christian's component notebook is still in progress. The following issues were raised in team review and should be resolved before his outputs are used:*
> - **`area_sqmi` unit bug:** with `TARGET_CRS = "EPSG:3310"` (meters), `geometry.area` is in m². Dividing by `FT_PER_MILE**2` mixes units (result ~10.76× too small). Fix: divide by `METERS_PER_MILE**2`.
> - **`planning_name_col` NameError (Section 7):** the line `planning_areas.rename(columns={planning_name_col: "planning_area"})` references an undefined variable. The CPNAME → planning_area rename above already handles it — delete the second rename.
> - **Coordinate sanity check:** confirm `longitude` ≈ −118 to −116 and `latitude` ≈ 32 to 33. If they appear as large numbers (hundreds of thousands), they are projected X/Y mislabeled as lon/lat and must be fixed at the geocoding step. Verify with `df[["longitude","latitude"]].describe()`.


## 8. Descriptive Statistics

Exploratory summary of the geocoded business-point dataset. (The tract-level descriptive statistics — ACS employment, LODES workplace jobs, spatial autocorrelation context — are in Canyu's component notebook.)

The code cells below load Joseph's exported indicator output and summarize it.


In [ ]:
# ── Load Joseph's exported indicator output ──────────────────────────────────
# Run Code_Joseph.ipynb first to produce joseph_indicators_output.geojson

gdf = gpd.read_file("joseph_indicators_output.geojson")
print(f"Business points loaded: {len(gdf)}")
print(f"Columns: {gdf.columns.tolist()}")
print()

STUDY_AREAS = ["Otay Mesa", "Kearny Mesa", "Miramar", "Sorrento Valley"]
buf_cols    = ["buffer_quarter_mi", "buffer_half_mi", "buffer_one_mi"]

print("=== Business count by subarea ===")
print(gdf["subarea"].value_counts())
print()

print("=== Industry group by subarea ===")
print(gdf.groupby(["subarea","industry_group"]).size().unstack(fill_value=0))
print()

print("=== Distance to nearest POE (miles) by subarea ===")
print(
    gdf.groupby("subarea")["dist_nearest_poe_mi"]
    .agg(n="count", mean="mean", median="median", std="std", min="min", max="max")
    .round(3)
)


In [ ]:
# ── Buffer overlap summary ────────────────────────────────────────────────────
print("=== Freight corridor buffer overlap (% inside) by subarea ===")
for col in buf_cols:
    print(f"\n{col}:")
    print(
        gdf.groupby("subarea")[col]
        .agg(n="count", n_inside="sum",
             pct_inside=lambda x: round(100*x.mean(), 1))
    )


In [ ]:
# ── POE distance histogram by study area ─────────────────────────────────────
AREA_COLORS = {
    "Otay Mesa":"#e8522a","Kearny Mesa":"#2c7bb6",
    "Miramar":"#1a9641","Sorrento Valley":"#7b3294"
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, area in enumerate(STUDY_AREAS):
    sub = gdf[gdf["subarea"] == area]["dist_nearest_poe_mi"].clip(upper=40)
    axes[i].hist(sub, bins=30, color=AREA_COLORS[area], alpha=0.8, edgecolor="white")
    axes[i].axvline(sub.median(), color="black", linestyle="--", linewidth=1.5,
                    label=f"Median: {sub.median():.2f} mi")
    axes[i].set_title(f"{area} (n={len(sub):,})", fontweight="bold")
    axes[i].set_xlabel("Distance to nearest POE (miles)")
    axes[i].set_ylabel("Count")
    axes[i].legend(fontsize=9)
    axes[i].spines[["top","right"]].set_visible(False)
fig.suptitle("Distribution of Distance to Nearest POE by Study Area",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Analysis

### Workflow

```
                  ┌─────────────────────────────────────────────┐
                  │  Census ACS + LODES + TIGER + SANDAG + GADM  │
                  └─────────────────────────────────────────────┘
                                     │
         ┌───────────────────────────┼───────────────────────────┐
         ▼                           ▼                           ▼
 [Canyu — tracts]          [Joseph — business pts]      [Christian — business pts]
  ACS employment            POE distance                 distance to freight roads
  LODES workplace jobs      freight buffers              (in progress)
  zoning/land-use overlay   Mann-Whitney + Chi-sq
  proximity indicators      maps + charts
         │                           │                           │
         └───────────────────────────┴───────────────────────────┘
                                     ▼
                          Combined StoryMap + Report
```

### Step-by-Step (Joseph's component — runnable below)

**Step 1 — CRS:** EPSG:3310 (California Albers, meters). Distances computed in meters, reported in miles.  
**Step 2 — Subarea assignment:** `gpd.sjoin` on Community Plan districts (Otay Mesa = OTAY MESA + OTAY MESA-NESTOR; Kearny Mesa; Miramar = MIRAMAR RANCH NORTH + SCRIPPS MIRAMAR RANCH; Sorrento Valley = MIRA MESA).  
**Step 3 — Indicator 1:** Euclidean distance to nearest POE.  
**Step 4 — Indicator 1b:** `gpd.sjoin_nearest` to nearest freight road.  
**Step 5 — Indicator 2:** binary inside/outside flags for 0.25/0.50/1.00-mile road buffers.  
**Step 6 — Statistics:** pairwise Mann-Whitney U on POE distance + Chi-square on buffer flags, with 95% bootstrap CIs.

### Tract-level component (Canyu)
ACS employment (context) + LODES workplace jobs (primary activity measure) joined to tracts; proximity indicators (`dist_poe_mi`, `dist_road_mi`, corridor area-share); zoning and general-plan land-use overlays; per-area comparison and county-wide association.

### ⚠️ Christian's component (in progress)
> *Paste Christian's analysis steps and code here once finalized — distance-to-freight-road indicator at the business-point level, with the unit and coordinate fixes from Section 7 applied.*

### Evolution from Proposal
Industrial zone polygons → business points + tracts. ESRI GeoEnrichment → Census LODES. Statistical significance testing added.


In [ ]:
# ── Analysis parameters (Joseph's component) ─────────────────────────────────
CRS_PROJ        = "EPSG:3310"    # California Albers (meters)
METERS_PER_MILE = 1609.34
BUFFER_M        = {"buffer_quarter_mi":402, "buffer_half_mi":805, "buffer_one_mi":1609}

STUDY_AREA_MAP = {
    "OTAY MESA":             "Otay Mesa",
    "OTAY MESA-NESTOR":      "Otay Mesa",
    "KEARNY MESA":           "Kearny Mesa",
    "MIRAMAR RANCH NORTH":   "Miramar",
    "SCRIPPS MIRAMAR RANCH": "Miramar",
    "MIRA MESA":             "Sorrento Valley",
}

print("Analysis parameters:")
print(f"  CRS: {CRS_PROJ}")
print(f"  Buffer distances (m): {BUFFER_M}")
print("\nFull analysis: run Code_Joseph.ipynb and Code_canyu_June_7th.ipynb")


## 10. Summary of Results

### Key Findings

**Indicator 1 — Distance to Nearest POE (business points, Joseph)**

| Subarea | n | Median (mi) |
|---------|---|-------------|
| Otay Mesa | 493 | 1.09 |
| Kearny Mesa | 206 | 20.75 |
| Miramar | 52 | 25.99 |
| Sorrento Valley | 369 | 24.99 |

Otay Mesa businesses are **19–24× closer** to a port of entry than businesses in the comparison areas.

**Indicator 2 — Freight Corridor Buffer Overlap**

| Buffer | Otay Mesa | Kearny Mesa | Miramar | Sorrento Valley |
|--------|-----------|-------------|---------|-----------------|
| 0.25 mi | 40.6% | 41.3% | 25.0% | 3.3% |
| 0.50 mi | 81.7% | 90.3% | 48.1% | 7.3% |
| 1.00 mi | 98.6% | 100.0% | 65.4% | 17.6% |

**Core finding:** POE proximity is the dominant spatial signal distinguishing Otay Mesa. Freight-road corridor proximity does **not** uniquely distinguish Otay Mesa — Kearny Mesa matches or exceeds it at every buffer distance — which confirms that not every freight indicator separates the border zone. It is specifically *port* proximity, not general road access, that marks Otay Mesa as distinctive.

*(Re-run the code cells in Section 8 / Code_Joseph.ipynb to regenerate these tables from current data.)*


## 11. Discussion

### Findings in Context of Literature
The strong POE-proximity clustering is consistent with border-zone industrial location research showing that freight-oriented businesses self-select for locations minimizing cross-border transport costs. The finding refines that literature: the border crossing itself — not road access generally — is the organizing principle. Kearny Mesa, an inland aerospace/defense cluster, sits as close to (or closer to) major freight roads than Otay Mesa, yet is ~20 miles from any POE. This shows freight-road proximity is necessary-but-not-distinctive, while POE proximity is the distinguishing feature.

### Trade-offs and Decision Points

**Buffer distance:** The 0.25 / 0.50 / 1.00-mile thresholds are researcher-defined. Reporting three thresholds (rather than one) addresses the instructor's concern that findings should not depend on a single arbitrary distance.

**CRS:** Joseph's component uses EPSG:3310 (meters) to match Christian's pipeline; Canyu's uses EPSG:2230 (US survey feet). Both are valid projected CRSs for San Diego; each reports final distances in miles. The **`area_sqmi` unit bug** flagged in Section 7 is a direct example of why the projection's native unit must be tracked carefully when computing area.

**Activity measure:** LODES workplace jobs (where work occurs) replaced ESRI GeoEnrichment and is preferred over ACS residence-based employment for locating industrial activity.

**Geocoding validity:** Geocode scores reflect address-matching confidence, not operational-location accuracy — the most significant unresolved concern (see Section 9c in Canyu's notebook).

**Subarea definition:** Official Community Plan districts replace researcher-defined circles/boxes, grounding subareas in city geography. Sorrento Valley has no official district; MIRA MESA is the closest match.


## 12. Conclusions & Future Work

### Did We Answer the Research Question?
Yes, with caveats. The spatial association between Otay Mesa's industrial activity and port-of-entry proximity is statistically significant and extremely strong. Critically, the multi-area comparison shows the association is specific to *ports*, not freight roads in general — strengthening rather than weakening the conclusion. The analysis documents where businesses are *registered*; operational locations may differ.

### Future Work

1. **Border-logistics indicator businesses** — identify NAICS subcategories unique to Otay Mesa (customs brokers 541614, freight forwarders 488510, cold storage 493120) to build a "border-economy index."
2. **Operational location validation** — extend the address-validity worksheet (Canyu §9c) to a larger sample using company websites / satellite imagery.
3. **Sensitivity analysis** — test a wider range of buffer distances to confirm robustness.
4. **Tijuana-side analysis** — incorporate maquiladora zones across the border.
5. **Drive-time distance** — replace straight-line POE distance with network/drive-time distance and border wait times.
6. **Otay Mesa East POE** — model the expected industrial shift if the proposed crossing opens.

### Expected Use
Most immediately relevant to SANDAG and City of San Diego planners considering land use near the proposed Otay Mesa East crossing. The framework extends to other US–Mexico crossings in California and Texas. Industry reviewer Connor Jennings (cc'd by the instructor) may suggest specific real-world application contexts.

---
*Notebook prepared for GPEC 447, UC San Diego, Spring 2026.*
